# The heat equation forgets its initial state

Section 6.2 of the notes.

The adjoint route of Section 6.2 recovers the state by solving the ODE backwards
in time. For a dissipative equation this fails. This notebook solves the heat
equation from four very different initial states and shows that they all end at
the same state, so that running time backwards would have to tell them apart
again.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

## 1. The heat equation on a grid

$u_t = D\,u_{xx}$ on $(-1,1)$ with $u(-1,t) = u(1,t) = 0$. On the interior grid
points $x_j = -1 + j\,\Delta x$, $j = 1,\dots,d$, with $\Delta x = 2/(d+1)$, the
second difference turns it into $\mathrm{d}\boldsymbol u/\mathrm{d}t = D\boldsymbol L\boldsymbol u$,
as in Section 2.5, with zero values outside the grid in place of the periodic
corners. $D\boldsymbol L$ is symmetric with negative eigenvalues $\lambda_k$ and
orthonormal eigenvectors $\boldsymbol v_k$, so the solution is exact:
$\boldsymbol u(t) = \sum_k e^{\lambda_k t}\,(\boldsymbol v_k^\top\boldsymbol u_0)\,\boldsymbol v_k$.
Every component decays, the fast ones much faster than the slow ones.

In [ ]:
def heat_operator(d, D):
    # Interior grid of (-1, 1), and the matrix D L with zero boundary values.
    x = np.linspace(-1, 1, d + 2)[1:-1]
    dx = x[1] - x[0]
    L = (
        np.diag(-2.0 * np.ones(d))
        + np.diag(np.ones(d - 1), 1)
        + np.diag(np.ones(d - 1), -1)
    ) / dx**2
    lam, V = np.linalg.eigh(D * L)
    return x, lam, V


def heat_solution(u0, t, lam, V):
    # u(t) for every initial state (rows of u0) and every time in t: (states, times, d).
    coef = u0 @ V
    return np.einsum("ik,tk,jk->itj", coef, np.exp(np.outer(t, lam)), V)


x, lam, V = heat_operator(d=199, D=1.0)
print(f"slowest rate {-lam.max():.3f}  (pi^2/4 = {np.pi**2 / 4:.3f})")
print(f"fastest rate {-lam.min():.0f}")

## 2. Four initial states

A smooth bump, a step, a narrow spike, and random noise.

In [ ]:
rng = np.random.default_rng(0)
u0 = np.stack(
    [
        1 - x**2,
        np.where(np.abs(x) < 0.5, 1.0, 0.0),
        np.exp(-((x / 0.03) ** 2)),
        rng.uniform(-1, 1, x.size),
    ]
)
names = ["bump", "step", "spike", "noise"]
t = np.linspace(0, 5, 1001)
u = heat_solution(u0, t, lam, V)
size = np.abs(u).max(axis=2)  # max over x of |u|, for each state and time
for name, s in zip(names, size):
    print(
        f"{name:6s} max|u| at t = 0, 0.1, 1, 5: "
        + "  ".join(f"{s[np.argmin(np.abs(t - tt))]:.1e}" for tt in (0, 0.1, 1, 5))
    )

In [ ]:
colors = ["#2a78d6", "#eb6834", "#8e3fb8", "#8a8983"]
styles = ["-", (0, (4, 2)), (0, (1, 1.2)), "-"]
plt.rcParams.update(
    {
        "font.family": "STIXGeneral",
        "mathtext.fontset": "stix",
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
fig, axes = plt.subplots(1, 3, figsize=(4.6, 1.6), gridspec_kw=dict(wspace=0.45))
i01 = np.argmin(np.abs(t - 0.1))
i1 = np.argmin(np.abs(t - 5.0))
for k in [3, 0, 1, 2]:  # the noise first, under the others
    axes[0].plot(
        x,
        u0[k],
        color=colors[k],
        ls=styles[k],
        lw=0.6 if k == 3 else 1.3,
        alpha=0.6 if k == 3 else 1.0,
        label=names[k],
    )
    axes[1].plot(x, u[k, i01], color=colors[k], ls=styles[k], lw=1.3)
    axes[2].plot(x, u[k, i1], color=colors[k], ls=styles[k], lw=1.3)
axes[0].set(xlabel="$x$", ylabel="$u$", title="$t = 0$", ylim=(-1.1, 1.1))
axes[1].set(xlabel="$x$", title="$t = 0.1$", ylim=(-1.1, 1.1))
axes[2].set(xlabel="$x$", title="$t = 5$", ylim=(-1.1, 1.1))
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.98),
    ncol=4,
    frameon=False,
)
fig.savefig(FIGURES / "heat_forgets.pdf", bbox_inches="tight")
plt.show()

## 3. Running time backwards

Backwards in time each component is multiplied by $e^{-\lambda_k t}$ instead of
$e^{\lambda_k t}$. A rounding error of size $10^{-16}$ in the fastest component
is multiplied by $e^{-\lambda_{\min} t}$.

In [ ]:
for back in (1e-4, 1e-3, 1e-2):
    print(
        f"backwards over t = {back:.0e}: fastest component grows by e^{-lam.min() * back:.0f}"
        f" = 10^{-lam.min() * back / np.log(10):.0f}"
    )